# Setup

Goal: rebuild the client, index and search tool from the intro notebook in as few cells as possible.

## Load the API key

In [ ]:
from dotenv import load_dotenv, find_dotenv

# walk up from this notebook's folder until a .env is found
env_path = find_dotenv(usecwd=True)
assert env_path, ".env not found in this folder or any parent"
load_dotenv(env_path)

## Smoke test the client

In [ ]:
from openai import OpenAI

openai_client = OpenAI()

response = openai_client.responses.create(
    model="gpt-4o-mini",
    input="Tell me a joke"
)

print(response.output_text)

## Load, chunk and index the Evidently docs

Same pipeline as before, collapsed into one cell.

In [ ]:
import json

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex


reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

## Search function and tool schema

In [ ]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}

# Handle tool calls generically

Goal: stop running each tool call by hand and let a helper turn any call into its output message.

## Dispatch helper

`make_call` looks up the tool by name, runs it with the parsed arguments, and wraps the result with the matching `call_id`. Only `search` is wired up for now.

In [ ]:
def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    # elif name == 'add_entry':
    #     result = add_entry(**arguments)
    else: 
        result = 'not found tool "{name}"'
    
    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

## Ask for several searches

The instructions now push the model to search at least three times and to leave "evidently" out of the queries.

In [ ]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

IMPORTANT: When you explore the knowledge base, make at least 3 different
searchers to make sure you explore the topic well.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

## Question and history

In [ ]:
question = "How do I create a dahsbord in Evidently?"

In [ ]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

## First call

In [ ]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=message_history,
    tools=[search_tool],
)

## Run every requested call

The whole `response.output` goes into the history, then each function call in it gets executed and its output appended.

In [ ]:
message_history.extend(response.output)

In [ ]:
message_history

In [ ]:
for message in response.output:
    if message.type == 'function_call':
        print(f'executing {message.name}({message.arguments})...')
        tool_call_output = make_call(message)
        message_history.append(tool_call_output)

The model asked for three searches in a single response, all with different wording - one call, three tool requests.

## Second call - the answer

In [ ]:
len(message_history)

In [ ]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=message_history,
    tools=[search_tool],
)

response.usage.input_tokens

About 12,000 input tokens: three sets of search results are now in the history.

In [ ]:
response.output[0].type

`message` rather than `function_call` - the model has enough and answers instead of asking for more.

In [ ]:
print(response.output_text)

# The agent loop

Goal: keep calling the model until it stops asking for tools, instead of counting the turns by hand.

## Instructions that plan in steps

The model is asked to search once, then twice more based on what it found, and to explain each search before making it.

In [ ]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

## Run until there are no tool calls

Each pass sends the history, records the output, runs any function calls, and prints any text. When a response has no function calls the model is done and the loop breaks.

In [ ]:
question = "How do I create a dahsbord in Evidently?"

In [ ]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=[search_tool],
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()
    
    if not has_function_calls:
        break

Three iterations, as instructed: one search, two follow-up searches, then the answer. The explanations and the exact queries change between runs.

# A tool that writes

Goal: give the agent a second tool that changes state, not just reads it.

## `add_entry` and its schema

It appends a new document to the same index, so later searches can find it.

In [ ]:
def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry"
            },
            "title": {
                "type": "string",
                "description": "The title of the documentation entry"
            },
            "description": {
                "type": "string",
                "description": "A short description summarizing the entry"
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry"
            }
        },
        "required": [
            "filename",
            "title",
            "description",
            "content"
        ]
    }
}

## Register the new tool

Same dispatcher with the `add_entry` branch switched on.

In [ ]:
def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    elif name == 'add_entry':
        result = add_entry(**arguments)
    else: 
        result = 'not found tool "{name}"'
    
    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

## Rerun the loop with both tools

In [ ]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]
tools = [search_tool, add_entry_tool]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()
    
    if not has_function_calls:
        break

## Ask the agent to save its answer

A follow-up turn on the same history - no new question, just an instruction.

In [ ]:
message_history.append(
    {"role": "user", "content": "add this content to our database"}
)

## Continue the loop

In [ ]:
while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()
    
    if not has_function_calls:
        break

The counter carries on from 4 because `iteration_number` was not reset. The model picked the `filename`, title and description itself - the filename is made up, not a real docs page.

## Check the index

The last document in the index is the one the agent just wrote.

In [ ]:
index.docs[-1]